# 🩺 Day 2: QLoRA Fine-Tuning LLaVA-1.5-7B on VQA-Med-2019
### Topic 4: Medical Diagnostic Support Visual Question Answering
> **Sprint Milestone:** D2-T1 (Launch QLoRA Fine-Tuning on Colab T4)  
> **Hardware Budget:** 1x NVIDIA T4 GPU (~15 GB VRAM)  
> **Technique:** QLoRA (4-bit NF4 Quantization + LoRA $r=16, \alpha=32$)  
> **Safety Feature:** Auto-saves checkpoints to Google Drive (with local fallback & 1-click zip download)

---## 1. 🖥️ Step 1: GPU Sanity Check & Dependency Installation
Check that Google Colab has assigned an **NVIDIA T4 GPU** and install the validated multi-modal training dependencies.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Connected to: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    raise SystemError("⚠️ No GPU detected! Go to: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU")

In [ ]:
# Install validated dependencies for LLaVA QLoRA fine-tuning
!pip install -q --upgrade pip
!pip install -q transformers>=4.37.0 peft>=0.7.0 bitsandbytes>=0.41.0 accelerate>=0.26.0
!pip install -q datasets torchvision pillow matplotlib sacrebleu rouge-score
print("✅ Dependencies successfully installed!")

---## 2. 📁 Step 2: Flexible Workspace Setup & Checkpoint Destination
Choose where to save your checkpoints and how to obtain the dataset:
* **Google Drive (Recommended):** If mounted, checkpoints survive Colab runtime disconnects.
* **Local Fallback:** If you skip Drive, checkpoints are stored in `/content/checkpoints/` (a 1-click cell at the end lets you download them as a zip).

In [ ]:
import os

MOUNT_GOOGLE_DRIVE = True  # Set to False if you do not want to use Google Drive

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/diagnostic_support/checkpoints/llava-med-qlora'
        print("✅ Google Drive mounted successfully!")
    except Exception as e:
        print(f"⚠️ Drive mount skipped or failed: {e}")
        OUTPUT_DIR = '/content/checkpoints/llava-med-qlora'
else:
    OUTPUT_DIR = '/content/checkpoints/llava-med-qlora'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"💾 Checkpoints will be saved to: {OUTPUT_DIR}")

### 2.1 Code & Dataset Acquisition Options
Run **one** of the methods below to ensure `data/processed/train.json`, `val.json`, and images are available in `/content`.

In [ ]:
# Option A: Clone repository from GitHub (Recommended if repository is pushed)
# !git clone https://github.com/TranQuan231005/diagnostic_support.git /content/diagnostic_support
# %cd /content/diagnostic_support

# Option B: Create directories locally if running as a standalone Colab notebook
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/raw/images", exist_ok=True)
print("Directory structure initialized:")
!ls -la data

---## 3. 📊 Step 3: Dataset Loading & Dual-Mode Execution
We provide a `QUICK_SANITY_RUN` toggle:
* `QUICK_SANITY_RUN = True` (Default): Trains on 200 samples (~5–10 mins) to verify that loss converges and checkpoints save without OOM.
* `QUICK_SANITY_RUN = False`: Trains on all 12,792 samples across 3 epochs.

In [ ]:
import json
import random
from PIL import Image

# ====================================================================
# CONFIGURATION TOGGLE
# ====================================================================
QUICK_SANITY_RUN = True   # Set to False for full 12,792 sample training
SANITY_TRAIN_SAMPLES = 200
SANITY_VAL_SAMPLES = 50
# ====================================================================

train_json_path = "data/processed/train.json"
val_json_path = "data/processed/val.json"

# If dataset JSONs exist on disk, load them; otherwise create synthetic demonstrator samples
if os.path.isfile(train_json_path) and os.path.isfile(val_json_path):
    with open(train_json_path, "r", encoding="utf-8") as f:
        raw_train_data = json.load(f)
    with open(val_json_path, "r", encoding="utf-8") as f:
        raw_val_data = json.load(f)
    print(f"Loaded existing dataset: {len(raw_train_data):,} train, {len(raw_val_data):,} val")
else:
    print("Creating self-contained synthetic medical demonstrator dataset for standalone Colab run...")
    categories = ["Modality", "Plane", "Organ System", "Abnormality"]
    modalities = ["CT", "MRI", "X-Ray", "Ultrasound"]
    planes = ["Axial", "Sagittal", "Coronal"]
    organs = ["Brain", "Lung", "Heart", "Liver"]
    abnormalities = ["Cardiomegaly", "Pneumothorax", "Nodule", "Effusion"]

    raw_train_data = []
    for i in range(250):
        img_name = f"synpic_{i:04d}.jpg"
        img_path = os.path.join("data/raw/images", img_name)
        if not os.path.isfile(img_path):
            img = Image.new("RGB", (336, 336), color=(i % 120 + 40, i % 120 + 40, i % 120 + 40))
            img.save(img_path)
        cat = categories[i % 4]
        if cat == "Modality":
            q, a = "What imaging modality is depicted?", modalities[i % len(modalities)]
        elif cat == "Plane":
            q, a = "In what plane is this image taken?", planes[i % len(planes)]
        elif cat == "Organ System":
            q, a = "What organ is principally shown?", organs[i % len(organs)]
        else:
            q, a = "What abnormality is observed in this scan?", abnormalities[i % len(abnormalities)]
        
        raw_train_data.append({
            "id": f"sample_{i:04d}",
            "image": img_path,
            "conversations": [
                {"from": "human", "value": f"<image>\n{q}"},
                {"from": "gpt", "value": a}
            ],
            "category": cat
        })
    raw_val_data = raw_train_data[:50]

if QUICK_SANITY_RUN:
    train_data = raw_train_data[:SANITY_TRAIN_SAMPLES]
    val_data = raw_val_data[:SANITY_VAL_SAMPLES]
    print(f"⚡ QUICK_SANITY_RUN enabled: Training on {len(train_data)} samples, evaluating on {len(val_data)} samples.")
else:
    train_data = raw_train_data
    val_data = raw_val_data
    print(f"🚀 FULL TRAINING enabled: Training on {len(train_data):,} samples.")

---## 4. 🧩 Step 4: Multi-Modal Data Collator & Prompt Formatter
In Vision-Language Models, the loss must be calculated **only on the model's answer**, not on the user prompt.
We construct a custom PyTorch `Dataset` and `DataCollator` that:
1. Loads and resizes the medical image.
2. Formats the conversational prompt: `USER: <image>\n{question}\nASSISTANT: {answer}`
3. Masks prompt token labels with `-100` so PyTorch CrossEntropyLoss ignores them.

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoProcessor

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
print(f"Loading AutoProcessor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

class VQAMedDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = item["image"]
        if os.path.isfile(img_path):
            image = Image.open(img_path).convert("RGB")
        else:
            image = Image.new("RGB", (336, 336), color=(128, 128, 128))
        
        # Extract human question and gpt answer
        q = item["conversations"][0]["value"].replace("<image>\n", "").strip()
        a = item["conversations"][1]["value"].strip()
        return {"image": image, "question": q, "answer": a}

class VQADataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        images = [b["image"] for b in batch]
        # Format prompt in standard LLaVA-1.5 conversational format
        prompts = [f"USER: <image>\n{b['question']}\nASSISTANT: {b['answer']}" for b in batch]
        prompt_only_texts = [f"USER: <image>\n{b['question']}\nASSISTANT:" for b in batch]

        # Process batch through processor
        inputs = self.processor(
            images=images,
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )

        # Create target labels and mask prompt tokens with -100
        labels = inputs["input_ids"].clone()
        for i, prompt_text in enumerate(prompt_only_texts):
            prompt_tokens = self.processor.tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            )["input_ids"]
            prompt_len = prompt_tokens.shape[1]
            labels[i, :prompt_len] = -100

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        inputs["labels"] = labels
        return inputs

train_dataset = VQAMedDataset(train_data)
val_dataset = VQAMedDataset(val_data)
data_collator = VQADataCollator(processor)
print(f"✅ Datasets initialized: {len(train_dataset)} train, {len(val_dataset)} val.")

---## 5. 🏗️ Step 5: Load LLaVA-1.5-7B in 4-bit NF4 & Attach LoRA Adapter
We configure:
1. **`BitsAndBytesConfig`**: 4-bit NF4 quantization (`bnb_4bit_compute_dtype=torch.float16`, double quant enabled).
2. **Freeze Vision Tower**: CLIP encoder remains completely frozen.
3. **Target Modules**: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` with $r=16, \alpha=32$.

In [ ]:
import torch
from transformers import BitsAndBytesConfig, LlavaForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading base model {MODEL_ID} in 4-bit NF4...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Enable gradient checkpointing for VRAM savings
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Configure LoRA targeting all attention and projection linear layers
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)

# Inspect trainable parameters
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in peft_model.parameters())
print("=" * 65)
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Total Parameters:     {all_params:,}")
print(f"Trainable Ratio:      {100.0 * trainable_params / all_params:.4f}%")
print(f"Current VRAM Usage:   {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
print("=" * 65)

---## 6. 🚀 Step 6: Trainer Configuration & Training Execution
Set hyperparameter arguments designed for Colab T4 stability:
* `per_device_train_batch_size = 2`
* `gradient_accumulation_steps = 4` (effective batch size = 8)
* `learning_rate = 2e-4` with cosine schedule
* `fp16 = True`
* Checkpoint saving every 50 or 100 steps.

In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback
import matplotlib.pyplot as plt

class LossPlotCallback(TrainerCallback):
    """Live callback to log training loss progression."""
    def __init__(self):
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append(logs["loss"])

loss_callback = LossPlotCallback()

num_epochs = 1 if QUICK_SANITY_RUN else 3
save_steps = 25 if QUICK_SANITY_RUN else 100

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=5 if QUICK_SANITY_RUN else 20,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=2,
    optim="paged_adamw_8bit",  # VRAM-saving 8-bit optimizer
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[loss_callback],
)

print("Starting QLoRA Fine-Tuning...")
train_result = trainer.train()
print("\n✅ Training completed successfully!")

In [ ]:
# Plot training loss curve
if loss_callback.losses:
    plt.figure(figsize=(8, 4))
    plt.plot(loss_callback.losses, marker="o", color="#2563eb", label="Training Loss")
    plt.title("VQA-Med-2019 QLoRA Training Loss Progression")
    plt.xlabel("Logged Steps")
    plt.ylabel("Cross Entropy Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plot_save_path = os.path.join(OUTPUT_DIR, "training_loss_curve.png")
    plt.savefig(plot_save_path, dpi=200)
    plt.show()
    print(f"Loss curve saved to: {plot_save_path}")

In [ ]:
# Save final LoRA adapter weights and processor
final_adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
peft_model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)
print(f"💾 Final adapter weights saved to: {final_adapter_dir}")
!ls -lh {final_adapter_dir}

---## 7. 🧪 Step 7: Checkpoint Validation & Sample Medical Inference
Verify that our fine-tuned adapter generates valid clinical answers on an unseen sample.

In [ ]:
from peft import PeftModel

# Test sample
test_sample = val_data[0]
test_img = Image.open(test_sample["image"]).convert("RGB") if os.path.isfile(test_sample["image"]) else Image.new("RGB", (336, 336), (128, 128, 128))
test_question = test_sample["conversations"][0]["value"].replace("<image>\n", "").strip()
ground_truth = test_sample["conversations"][1]["value"].strip()

prompt = f"USER: <image>\n{test_question}\nASSISTANT:"
inputs = processor(images=test_img, text=prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        temperature=0.0,
    )

generated_text = processor.decode(output_tokens[0], skip_special_tokens=True)
answer_part = generated_text.split("ASSISTANT:")[-1].strip()

print("=" * 65)
print(f"❓ Question:         {test_question}")
print(f"🎯 Ground Truth:     {ground_truth}")
print(f"🤖 Model Prediction: {answer_part}")
print("=" * 65)

---## 8. 📦 Step 8: Export Checkpoints (1-Click Zip Download or HF Hub)
If you ran the notebook without Google Drive, run the cell below to download the trained adapter directly to your local PC.

In [ ]:
# Zip adapter weights for 1-click download
import shutil
zip_path = "/content/llava_med_qlora_adapter"
shutil.make_archive(zip_path, 'zip', final_adapter_dir)
print(f"Created zip archive: {zip_path}.zip")

try:
    from google.colab import files
    print("Downloading adapter weights to your local machine...")
    files.download(f"{zip_path}.zip")
except Exception as e:
    print(f"Automatic browser download skipped: {e}. You can download {zip_path}.zip from the Colab file tree on the left.")

### Optional: Push Adapter to Hugging Face Hub

In [ ]:
# Uncomment and run to upload the adapter directly to your Hugging Face account
# !huggingface-cli login
# peft_model.push_to_hub("your-hf-username/llava-1.5-7b-vqamed-qlora", private=True)
# processor.push_to_hub("your-hf-username/llava-1.5-7b-vqamed-qlora", private=True)